# ZMQ Scheduler: Single Worker Queue

This notebook explains the simplest scheduler deployment.

One worker process owns one GPU and one preset. The client can submit many forecast jobs. The worker handles one job at a time. Later jobs wait in the ZMQ command socket until the current job finishes.

Use this pattern when one GPU should stay warm and receive jobs from a notebook, a script, or a web service.

Prerequisites:

- Run `docs/example_era5.ipynb` first, or make sure the ERA5 checkpoint and cache already exist under `ASSET_ROOT`.
- Use an absolute asset root on the data disk.
- This tutorial uses `download=False`, so it will not contact CDS.
- Always run the cleanup cell at the end. The worker is a real subprocess and owns GPU memory.

## Mental Model

There are two sockets.

- The command socket carries requests from the client to the worker.
- The event socket carries progress events from the worker back to the client.

A successful job emits this sequence:

`accepted` -> `preparing` -> `running` -> `step` -> `completed`

The important rule is simple: this notebook starts one worker, so only one job runs on the GPU at a time.

## What Is ZMQ

ZMQ, also called ZeroMQ, is a lightweight message transport library. It gives Python processes a socket API for sending bytes to each other. In this project, those bytes are JSON-encoded scheduler commands and events.

The scheduler uses two one-way channels. The client sends forecast commands through a `PUSH` socket. The worker receives them through a `PULL` socket. The worker sends progress events through another `PUSH` socket. The client receives them through another `PULL` socket.

The important point is that the client and worker do not need to live in the same Python process. They can be in the same notebook process, in a local subprocess, or in separate terminals. If the address starts with `ipc://`, ZMQ uses a local inter-process socket file. If the address starts with `tcp://`, ZMQ uses a TCP port.

This tutorial uses `ipc://` because all processes run on the same machine. A production service often uses `tcp://` so other processes can connect through a stable host and port.

In [1]:
import os

if os.environ.get("OMP_NUM_THREADS") in (None, "", "0"):
    os.environ["OMP_NUM_THREADS"] = "1"

import subprocess
import sys
import time
from pathlib import Path

import zmq

from flash_aurora.engine.core.asset_root import normalize_asset_root
from flash_aurora.engine.core.redaction import safe_path
from flash_aurora.scheduler import ForecastClient, ForecastClientConfig, ForecastRequest
from flash_aurora.scheduler.protocol import ForecastEvent
from flash_aurora.scheduler.worker import wait_for_bind

PRESET = "era5_pretrained"
VALID_TIME = "2023-01-01T06:00:00"
TIME_INDEX = 1
ROLLOUT_STEPS = 2
INFERENCE_PRECISION = "bf16_mixed@fp32"

ASSET_ROOT: Path | str | None = Path("/root/autodl-tmp/aurora")
ASSET_ROOT = normalize_asset_root(ASSET_ROOT)

SOCKET_DIR = ASSET_ROOT / "scheduler_single_worker"
SOCKET_DIR.mkdir(parents=True, exist_ok=True)
COMMAND_ADDR = f"ipc://{SOCKET_DIR / 'commands.ipc'}"
EVENT_ADDR = f"ipc://{SOCKET_DIR / 'events.ipc'}"
EXPORT_DIR = ASSET_ROOT / "output" / "scheduler_single_worker"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("asset_root:", safe_path(ASSET_ROOT))
print("command_addr:", COMMAND_ADDR)
print("event_addr:", EVENT_ADDR)

asset_root: ~/autodl-tmp/aurora
command_addr: ipc:///root/autodl-tmp/aurora/scheduler_single_worker/commands.ipc
event_addr: ipc:///root/autodl-tmp/aurora/scheduler_single_worker/events.ipc


In [2]:
WORKER_JOIN_TIMEOUT_S = 120.0


def spawn_worker() -> subprocess.Popen[bytes]:
    cmd = [
        sys.executable,
        "-m",
        "flash_aurora.scheduler",
        "--preset",
        PRESET,
        "--asset-root",
        str(ASSET_ROOT),
        "--worker-id",
        "single-era5",
        "--device",
        "cuda:0",
        "--capacity",
        "1",
        "--inference-precision",
        INFERENCE_PRECISION,
        "--command-addr",
        COMMAND_ADDR,
        "--event-addr",
        EVENT_ADDR,
        "--poll-timeout-ms",
        "500",
    ]
    return subprocess.Popen(cmd)


def print_event(event: ForecastEvent) -> None:
    if event.kind == "step" and event.export_path is not None:
        print(f"{event.request_id}: step {event.step}: {safe_path(event.export_path)}")
        return
    if event.kind == "step" and event.valid_time is not None:
        print(f"{event.request_id}: step {event.step} valid_time={event.valid_time}")
        return
    print(f"{event.request_id or 'worker'}: {event.kind}")


def shutdown_client(client: ForecastClient | None) -> None:
    if client is None:
        return
    try:
        client.shutdown_worker()
    finally:
        client.close()


def terminate_worker(worker_proc: subprocess.Popen[bytes] | None) -> None:
    if worker_proc is None or worker_proc.poll() is not None:
        return
    worker_proc.terminate()
    try:
        worker_proc.wait(timeout=WORKER_JOIN_TIMEOUT_S)
    except subprocess.TimeoutExpired:
        worker_proc.kill()
        worker_proc.wait(timeout=10)

## 1. Start One Worker

This cell starts one worker subprocess. The worker loads the engine lazily: it does not move the model to GPU until the first forecast job needs it.

The worker listens on `COMMAND_ADDR` and sends progress to `EVENT_ADDR`.

In [3]:
context = zmq.Context.instance()
worker_proc: subprocess.Popen[bytes] | None = spawn_worker()
wait_for_bind(COMMAND_ADDR)
print("worker started")

[worker] id=single-era5 preset=era5_pretrained device=cuda:0 capacity=1 command=ipc:///root/autodl-tmp/aurora/scheduler_single_worker/commands.ipc event=ipc:///root/autodl-tmp/aurora/scheduler_single_worker/events.ipc
worker started


## 2. Connect A Client

The client connects to the worker sockets. A health check proves that the worker is alive and that it is serving the expected preset.

In [4]:
client: ForecastClient | None = ForecastClient(
    ForecastClientConfig(
        command_addr=COMMAND_ADDR,
        event_addr=EVENT_ADDR,
        recv_timeout_ms=600_000,
    ),
    context=context,
)

health = client.health()
print("kind:", health.kind)
print("worker_preset:", health.worker_preset)
print("worker_id:", health.worker_id)
print("worker_device:", health.worker_device)
print("worker_capacity:", health.worker_capacity)

if health.worker_preset != PRESET:
    raise RuntimeError(f"Expected preset {PRESET!r}, got {health.worker_preset!r}")

kind: health
worker_preset: era5_pretrained
worker_id: single-era5
worker_device: cuda:0
worker_capacity: 1


## 3. Submit Two Jobs Before Reading Events

This is the queue demonstration.

The client sends two forecast requests immediately. The worker receives both commands, but it can only run one GPU rollout at a time. Therefore the second request waits until the first request emits `completed`.

The output order should make this visible: all events for `single-job-1` complete before `single-job-2` starts running.

In [5]:
requests = [
    ForecastRequest(
        request_id="single-job-1",
        preset=PRESET,
        steps=ROLLOUT_STEPS,
        valid_time=VALID_TIME,
        time_index=TIME_INDEX,
        download=False,
        export_dir=str(EXPORT_DIR / "single-job-1"),
    ),
    ForecastRequest(
        request_id="single-job-2",
        preset=PRESET,
        steps=1,
        valid_time=VALID_TIME,
        time_index=TIME_INDEX,
        download=False,
        output_mode="metadata_only",
    ),
]

assert client is not None
for request in requests:
    client.submit(request)
    print("submitted:", request.request_id)

for request in requests:
    print(f"\n--- events for {request.request_id} ---")
    for event in client.events(request.request_id):
        print_event(event)

submitted: single-job-1
submitted: single-job-2

--- events for single-job-1 ---
single-job-1: accepted
single-job-1: preparing
single-job-1: running
single-job-1: step 0: ~/autodl-tmp/aurora/output/scheduler_single_worker/single-job-1/prediction-000.nc
single-job-1: step 1: ~/autodl-tmp/aurora/output/scheduler_single_worker/single-job-1/prediction-001.nc
single-job-1: completed

--- events for single-job-2 ---
single-job-2: accepted
single-job-2: preparing
single-job-2: running
single-job-2: step 0 valid_time=2023-01-01T12:00:00
single-job-2: completed


## 4. Cleanup

This cell is not optional.

It sends `shutdown` to the worker, closes the client sockets, and terminates the subprocess if it did not exit cleanly. This is the step that prevents a hidden Python process from holding GPU memory.

In [6]:
shutdown_client(client)
terminate_worker(worker_proc)
time.sleep(0.1)
print("single worker scheduler closed")

single worker scheduler closed
